In [0]:
%fs ls abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices

In [0]:
%sql
CREATE OR REPLACE TABLE deltacatalog.deltadb.invoices_ttv
AS 
SELECT * from PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet`;

In [0]:
%sql
SELECT * FROM deltacatalog.deltadb.invoices_ttv LIMIT 5;

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.invoices_ttv;

In [0]:
%sql
DELETE FROM deltacatalog.deltadb.invoices_ttv
where customer_id=1;

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.invoices_ttv;

In [0]:
%sql
UPDATE deltacatalog.deltadb.invoices_ttv
SET quantity = 25
where customer_id=5;

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.invoices_ttv;

In [0]:
%sql
INSERT INTO deltacatalog.deltadb.invoices_ttv
select * from PARQUET.`abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet`;

Above command is last transaction and that should be latest version of this table

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.invoices_ttv;

In [0]:
%sql
SELECT * from deltacatalog.deltadb.invoices_ttv
where customer_id=1;

In [0]:
%sql
SELECT * from deltacatalog.deltadb.invoices_ttv
where customer_id=5;

In [0]:
%sql
SELECT count(*) as countOfRows from deltacatalog.deltadb.invoices_ttv
where customer_id > 100;

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.invoices_ttv;

In [0]:
%sql
SELECT * FROM deltacatalog.deltadb.invoices_ttv
where customer_id=1;

In [0]:
%sql
SELECT * FROM deltacatalog.deltadb.invoices_ttv version as of 0
where customer_id=1;

In [0]:
%sql
SELECT * FROM deltacatalog.deltadb.invoices_ttv timestamp as of '2026-04-27T07:36:02.000+00:00' 
where customer_id=1;

Restoring to above version

In [0]:
%sql
RESTORE TABLE deltacatalog.deltadb.invoices_ttv TO VERSION AS OF 0;

In [0]:
%sql
select * from deltacatalog.deltadb.invoices_ttv 
where customer_id=1;

In [0]:
%sql
DESCRIBE HISTORY deltacatalog.deltadb.invoices_ttv;

- **Version 3** timestamp : **2026-04-27T 07:38:43.000+00:00**
- **Version 1** timestamp : **2026-04-27T 07:36:36.000+00:00**

What if I enter a timing which is : **_2026-04-27T 07:37:36.000+00:00_**, What will happen?

In [0]:
%sql
SELECT * FROM deltacatalog.deltadb.invoices_ttv TIMESTAMP AS OF '2026-04-27T07:37:36.000+00:00'
where customer_id in (1,5);
-- Will find latest file existed before this timestamp if this timestamp is not present.

In [0]:
%sql
SELECT * FROM deltacatalog.deltadb.invoices_ttv
where customer_id in (1,5);

In [0]:
df = spark.read.option("versionAsOf","1").table("deltacatalog.deltadb.invoices_ttv")

In [0]:
from pyspark.sql.functions import col

display(df.filter("customer_id=5")) #expr format
display(df.filter(col("customer_id")==5)) #function specific format.

In [0]:
from pyspark.sql.functions import col

df = spark.read.option("versionAsOf","2").table("deltacatalog.deltadb.invoices_ttv")
display(df.filter("customer_id=5")) #expr format
display(df.filter(col("customer_id")==5)) #function specific format.

In [0]:
from pyspark.sql.functions import col

df = spark.read.option("timestampAsOF","2026-04-27T07:38:43.000+00:00").table("deltacatalog.deltadb.invoices_ttv")
display(df.filter("customer_id=5")) #expr format
display(df.filter(col("customer_id")==5)) #function specific format.